# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection

**Complete 13-stage pipeline on CIC-IDS2017**, with a figure saved to `../figures/`
at every stage for IEEE paper use.

This notebook drives the tested `nids` package in `../src/nids` -- every
computation (data loading, session reconstruction, token encoding, model
training, fusion, pattern mining, the attack-state graph, the risk
meta-learner, streaming evaluation, explainability) reuses that package's
already-debugged, already-tuned code, rather than reimplementing it inline.
This notebook cell layer adds the requested visualizations and PNG exports
on top.

**Chronological, session-aware split** (Review-corrected -- never a random
shuffle): Train = Monday+Tuesday, Validation = Wednesday, Test =
Thursday+Friday. All 15 class labels are retained, including the two
structurally rare ones (Heartbleed, Infiltration) -- see the Stage 1 cell
for the zero-training-count caveat this split creates for them.

A few of this notebook's constants differ from an initial raw spec because
they were already empirically tuned/debugged earlier in this project (see
inline notes where that happens) -- e.g. `FPGROWTH_MIN_SUPPORT` /
`PREFIXSPAN_MIN_SUPPORT` are `0.02`, not `0.15`/`0.10`: at 0.15/0.10 almost
nothing attack-specific ever cleared the support bar on the real dataset
(see `config.py`'s comment for the measurement behind that change).

## Setup
1. `pip install -r ../requirements.txt`
2. Point `DATA_DIR` below at your local CIC-IDS2017 folder
   (default: `D:\IDSPROJECT2026\CIC-IDS2017`).
3. Run All Cells.


In [ ]:
# CELL 0: Environment Setup
# %pip install mlxtend shap xgboost imbalanced-learn networkx seaborn tensorflow torch  # uncomment if needed

import os
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from nids import (
    config, data_loading, preprocessing, features, sessions, events,
    models, fusion, pattern_mining, attack_graph, risk, streaming,
    explainability, metrics, pipeline, ablation,
)

# --- Global constants (named, not magic numbers) --------------------------
SEED = config.RANDOM_SEED  # 42, used by numpy/sklearn/torch throughout the nids package
np.random.seed(SEED)

# DATA_DIR: point this at your local CIC-IDS2017 folder. NIDS_DATA_DIR lets
# this notebook be re-run against a different folder (e.g. a synthetic
# smoke-test dataset) without editing the file.
DATA_DIR = Path(os.environ.get("NIDS_DATA_DIR", str(config.DATA_DIR)))
print(f"DATA_DIR = {DATA_DIR}")

VOCAB = config.TOKENS                 # the 10-token behavioral vocabulary
TOKEN2ID = config.TOKEN_TO_IDX
CLASSES = config.CLASSES              # all 15 class labels, K=15
K = config.NUM_CLASSES

FIGURES_DIR = Path("..") / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name: str):
    '''Every stage below calls this right before plt.show() so every figure
    is written to ../figures/<name>.png for IEEE paper use.'''
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=120, bbox_inches="tight")
    print(f"Saved {path}")

# --- Publication-quality matplotlib defaults -------------------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("tab20", n_colors=K)  # 15-class colour consistency
CLASS_COLOR = dict(zip(CLASSES, PALETTE))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print(f"Behavioral vocabulary ({len(VOCAB)} tokens): {VOCAB}")
print(f"Classes (K={K}): {CLASSES}")


## Stage 1 · Data Acquisition & Partition

Auto-discovers the 7-8 daily CIC-IDS2017 CSVs by weekday name, strips the
well-known leading-space column names (e.g. `" Label"` -> `label`, handled by
`data_loading.normalize_columns`/`normalize_label` for *every* column, not
just `Label`), tags each row with its source day, and applies the
**chronological, session-aware split**: Train=Mon+Tue, Val=Wed, Test=Thu+Fri.

**Read this before you interpret real-dataset results:** in the real
CIC-IDS2017 release, Heartbleed flows exist only in the Wednesday file and
Infiltration flows only in the Thursday file, so under this split both have
**zero training examples**. The diagnostic step below prints the full
`(label, day)` breakdown so you can verify this against whatever files you
actually loaded, and `pipeline.check_zero_shot_classes` warns loudly if it
applies to your run. Report it as an honest limitation, not a bug.


In [ ]:
# STAGE 1: Data Acquisition & Partition -- diagnostic step (run first)
raw_df = data_loading.load_raw_data(DATA_DIR)
label_day_counts = pipeline.print_full_dataset_diagnostics(raw_df)


In [ ]:
# STAGE 1 continued: clean + chronological split
train_df, val_df, test_df = pipeline.stage1_2_load_clean_split(DATA_DIR)
print("train/val/test flow counts:", len(train_df), len(val_df), len(test_df))

# OUTPUT: horizontal bar chart of class distribution for all 3 splits
dist = pd.DataFrame({
    "train": train_df[config.LABEL_COLUMN].value_counts(),
    "val": val_df[config.LABEL_COLUMN].value_counts(),
    "test": test_df[config.LABEL_COLUMN].value_counts(),
}).reindex(CLASSES).fillna(0)

fig, ax = plt.subplots(figsize=(9, 7))
dist.plot(kind="barh", ax=ax, width=0.75)
ax.set_xscale("log")
ax.set_xlabel("flow count (log scale)")
ax.set_title("Stage 1 - Class distribution across chronological splits")
ax.legend(title="split")
plt.tight_layout()
savefig("stage1_class_distribution.png")
plt.show()


## Stage 4 · Bidirectional Session Reconstruction

**Novel contribution**: a *symmetric* 5-tuple key
`(min(src_ip,dst_ip), max(src_ip,dst_ip), protocol, min(port), max(port))`
means the forward and reverse direction of the same conversation always hash
to the same session, instead of silently becoming two duplicate sessions.
Session timeout `tau = 60s` (`config.SESSION_TIMEOUT_SECONDS`): a gap larger
than `tau` between two flows on the same key starts a new session. A session
can never span a partition boundary (it is built independently per
train/val/test), so there is no leakage from this step.


In [ ]:
# STAGE 4: Bidirectional Session Reconstruction
train_df = pipeline.stage4_5_sessions_and_events(train_df)
val_df = pipeline.stage4_5_sessions_and_events(val_df)
test_df = pipeline.stage4_5_sessions_and_events(test_df)

train_summary = sessions.session_summary(train_df)
print(f"{len(train_df)} training flows -> {train_df['session_id'].nunique()} training sessions "
      f"(tau={config.SESSION_TIMEOUT_SECONDS:.0f}s)")

# OUTPUT: session size histogram (log y-axis) + sessions-per-class bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(train_summary["n_flows"], bins=30, color=PALETTE[0], edgecolor="white")
axes[0].set_yscale("log")
axes[0].set_xlabel("flows per session")
axes[0].set_ylabel("session count (log)")
axes[0].set_title("Session size distribution (train)")

sess_per_class = train_summary["label"].value_counts().reindex(CLASSES).fillna(0)
axes[1].bar(sess_per_class.index, sess_per_class.values,
            color=[CLASS_COLOR[c] for c in sess_per_class.index])
axes[1].set_yscale("log")
axes[1].set_ylabel("sessions (log)")
axes[1].set_title("Sessions per class (train)")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
savefig("stage4_sessions.png")
plt.show()


## Stage 5 · Behavioral Event Encoding

**Novel contribution**: every flow is mapped to exactly one of 10
attack-agnostic behavioral tokens using only flow attributes available
without the ground-truth label (TCP flags, packet/byte counts, destination
port, duration) -- this avoids the circularity of a rule that itself
requires knowing the attack label. The rule table below (`events.RULE_TABLE`,
priority order, first match wins) is published verbatim, matching the
review requirement that the vocabulary's construction be explicit and
reproducible. Its exact thresholds were tuned against a worked SSH
brute-force example and differ slightly from an initial raw-spec draft
(e.g. `AUTH_FAIL`/`AUTH_SUCCESS` use `SMALL_BYTES=500`/`LARGE_BYTES=5000`
cutoffs plus RST/FIN/ACK conditions, not a bare `total_bytes<2000` check) --
see `src/nids/events.py` docstring for why.


In [ ]:
# STAGE 5: Behavioral Event Encoding
print("Rule table (published verbatim):")
display(pd.DataFrame(events.RULE_TABLE, columns=["token", "rule"]))

# OUTPUT: token frequency bar chart
token_counts = train_df["token"].value_counts().reindex(VOCAB).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(token_counts.index, token_counts.values, color=sns.color_palette("tab20", len(VOCAB)))
axes[0].set_yscale("log")
axes[0].set_ylabel("flow count (log)")
axes[0].set_title("Behavioral token frequency (train)")
axes[0].tick_params(axis="x", rotation=75)

# OUTPUT: example token sequence step-plot for one auth-port (SSH/FTP-like) attack session
auth_attack_sessions = train_df[
    (train_df[config.LABEL_COLUMN].isin(["SSH-Patator", "FTP-Patator"]))
]["session_id"]
if len(auth_attack_sessions) > 0:
    example_sid = auth_attack_sessions.iloc[0]
    example_seq = events.build_session_sequences(train_df[train_df["session_id"] == example_sid])[example_sid]
    tok_ids = [TOKEN2ID[t] for t, _ in example_seq]
    axes[1].step(range(len(tok_ids)), tok_ids, where="mid", marker="o", color=PALETTE[3])
    axes[1].set_yticks(range(len(VOCAB)))
    axes[1].set_yticklabels(VOCAB, fontsize=8)
    axes[1].set_xlabel("flow index within session")
    axes[1].set_title(f"Example token sequence - session {example_sid}")
else:
    axes[1].text(0.5, 0.5, "No SSH/FTP-Patator session in this split", ha="center", va="center")
    axes[1].axis("off")

plt.tight_layout()
savefig("stage5_token_encoding.png")
plt.show()


## Stage 2 · Preprocessing & Imbalance Handling

Inf -> NaN -> drop, then features are clipped at the 99th percentile and
Min-Max scaled -- **fit on the training split only**, applied unchanged to
val/test (no refit -- Review §6, prevents distribution leakage from val/test
into the scaler).

- **Branch A -- Class Weights** (primary imbalance strategy):
  `W_c = N_train / (K * N_c)`, applied to RF, XGBoost and the LSTM's loss.
- **Branch B -- SMOTE-KNN** (optional, RF/XGBoost only): `SMOTE(k=5)` then
  `EditedNearestNeighbours(k=3)`, restricted to classes with
  `N_c >= config.SMOTE_MIN_CLASS_COUNT` (50) training samples. The LSTM,
  FP-Growth, PrefixSpan and the attack-state graph **never** receive
  synthetic samples (temporal-stream firewall -- a synthetic flow has no
  real timestamp, so it cannot be a real sequential/co-occurring event).
  This notebook uses **Branch A by default**; the SMOTE-KNN comparison cell
  is opt-in below.


In [ ]:
# STAGE 2/3: train-only preprocessing, class weights, feature groups
pre, X_train, X_val, X_test, class_weights, fsets = pipeline.stage2_3_preprocess_and_group(train_df, val_df, test_df)

print("Scaled feature matrix shapes:", X_train.shape, X_val.shape, X_test.shape)
print(f"RF features: {len(fsets['rf'])} | XGBoost features: {len(fsets['xgb'])} | "
      f"LSTM context features: {len(fsets.get('lstm_context', []))}")

# OUTPUT: side-by-side bar charts of class counts (log scale) and class weights
y_train = train_df[config.LABEL_COLUMN]
counts = y_train.value_counts().reindex(CLASSES).fillna(0)
weights_series = pd.Series(class_weights).reindex(CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(counts.index, counts.values, color=[CLASS_COLOR[c] for c in counts.index])
axes[0].set_yscale("log")
axes[0].set_ylabel("training flow count (log)")
axes[0].set_title("Stage 2 - Class counts (train, log scale)")
axes[0].tick_params(axis="x", rotation=90)

axes[1].bar(weights_series.index, weights_series.values, color=[CLASS_COLOR[c] for c in weights_series.index])
axes[1].set_ylabel(r"$W_c = N_{train} / (K \cdot N_c)$")
axes[1].set_title("Stage 2 - Class weights (Branch A)")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
savefig("stage2_imbalance.png")
plt.show()


In [ ]:
# STAGE 2 (optional): Branch B -- SMOTE-KNN comparison, RF/XGBoost feature sets only.
# NOT used by default (Branch A / class weighting is primary). This cell is
# purely a size comparison; it does not feed Stage 6 unless you explicitly
# pass use_smote_branch_b=True to pipeline.stage6_train_models below.
X_rf_smote, y_rf_smote = preprocessing.smote_knn_resample(X_train[fsets["rf"]], y_train)
print("Branch A (original) size:", len(X_train), " | Branch B (SMOTE-KNN) size:", len(X_rf_smote))
print("Branch B per-class counts (top 8):")
print(y_rf_smote.value_counts().head(8))


## Stage 3 · Feature Engineering & Group Assignment

The 78 CICFlowMeter features are partitioned into 4 domain-knowledge groups
(keyword-matched against normalized column names), each ranked by mutual
information against the training labels (`X_train` **without** SMOTE, i.e.
Branch A) within its own group:

| Group | Feature type | Consumer |
|---|---|---|
| A (~32) | flow-statistical (packet length stats, byte counts, IAT) | Random Forest |
| B (~18) | protocol / communication (ports, TCP flags, header lengths) | Shared (RF+XGBoost) |
| C (~15) | derived temporal (Flow Duration, IAT mean/std/max/min) | LSTM context |
| D (~13) | TCP behavioral flag counts (SYN/ACK/RST/FIN rates) | XGBoost |

RF trains on Group A+B, XGBoost on Group B+D (matching `fsets["rf"]` /
`fsets["xgb"]` built above).


In [ ]:
# STAGE 3: Feature Engineering & Group Assignment
# NOTE: use X_train's own (already variance-filtered, scaled) columns here,
# not the raw candidate_feature_columns(train_df) list -- TrainOnlyPreprocessor
# drops near-constant columns during fit(), so a handful of raw candidates
# are never present in X_train and would KeyError if indexed directly.
from sklearn.feature_selection import mutual_info_classif

feat_cols = list(X_train.columns)
groups = features.assign_feature_groups(feat_cols)
# features.rank_by_mutual_information returns ranked COLUMN NAMES (used
# internally by build_model_feature_sets, called inside stage2_3_preprocess_and_group
# above); for a score bar chart we need the actual MI values, computed the
# same way (discrete_features=False, same random_state) via sklearn directly.
mi_values = mutual_info_classif(X_train[feat_cols], y_train, discrete_features=False, random_state=SEED)
mi_scores = pd.Series(mi_values, index=feat_cols)

# OUTPUT: MI score bar chart (top 30) + donut chart of group sizes
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
top_mi = mi_scores.sort_values(ascending=False).head(30)
axes[0].barh(top_mi.index[::-1], top_mi.values[::-1], color=PALETTE[4])
axes[0].set_xlabel("mutual information")
axes[0].set_title("Stage 3 - Top 30 features by MI (train)")

group_sizes = {g: len(cols) for g, cols in groups.items()}
axes[1].pie(
    group_sizes.values(), labels=[f"Group {g} ({n})" for g, n in group_sizes.items()],
    colors=sns.color_palette("tab20", len(group_sizes)), wedgeprops=dict(width=0.45), startangle=90,
)
axes[1].set_title("Stage 3 - Feature group sizes")

plt.tight_layout()
savefig("stage3_feature_groups.png")
plt.show()


## Stage 6 · Multi-Model Parallel Detection

Three classifiers, each on its designated feature group, trained with
**Branch A class weights** (`use_smote_branch_b=False`). All three output
calibrated probability vectors `P ∈ R^15`; no hard-decision labels cross
between stages.

- **6a Random Forest** -- Group A+B tabular features, flow-level.
- **6b Bidirectional LSTM** -- session token sequences (never raw flow rows,
  never SMOTE-touched), session-level.
- **6c XGBoost** -- Group B+D tabular features, flow-level.


In [ ]:
# STAGE 6: train RF + XGBoost + BiLSTM (Branch A -- class-weighted, primary methodology)
trained = pipeline.stage6_train_models(X_train, train_df, fsets, class_weights, use_smote_branch_b=False)
print("Models trained. Branch:", trained.branch)


In [ ]:
# STAGE 6a: Random Forest -- flow-level evaluation on the validation split
from sklearn.metrics import roc_curve, auc as sk_auc
from sklearn.preprocessing import label_binarize

proba_rf_val = models.predict_proba_rf(trained.rf, X_val[fsets["rf"]])
pred_rf_val = np.array(CLASSES)[proba_rf_val.argmax(axis=1)]
y_val_true = val_df[config.LABEL_COLUMN].values

rf_report = metrics.per_class_report(y_val_true, pred_rf_val)
rf_macro_f1 = metrics.macro_f1(y_val_true, pred_rf_val)
print(f"RF validation macro-F1: {rf_macro_f1:.4f}")
display(rf_report)

# OUTPUT: confusion matrix heatmap + per-class F1 bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm = metrics.confusion(y_val_true, pred_rf_val)
sns.heatmap(cm, annot=False, cmap="viridis", ax=axes[0], cbar=True)
axes[0].set_title("Stage 6a - RF confusion matrix (validation)")
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")

rf_report_classes = rf_report[rf_report["class"] != "MACRO_AVG"]  # per_class_report appends a MACRO_AVG summary row
axes[1].bar(rf_report_classes["class"], rf_report_classes["f1"], color=[CLASS_COLOR[c] for c in rf_report_classes["class"]])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("F1")
axes[1].set_title("Stage 6a - RF per-class F1 (validation)")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
savefig("stage6a_rf_results.png")
plt.show()

# OUTPUT: ROC curves (one-vs-rest), only for classes present in y_val_true
present = [c for c in CLASSES if (y_val_true == c).any()]
y_val_bin = label_binarize(y_val_true, classes=CLASSES)
fig, ax = plt.subplots(figsize=(8, 7))
for c in present:
    i = config.CLASS_TO_IDX[c]
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], proba_rf_val[:, i])
    ax.plot(fpr, tpr, label=f"{c} (AUC={sk_auc(fpr, tpr):.2f})", color=CLASS_COLOR[c])
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Stage 6a - RF ROC curves, one-vs-rest (validation)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
savefig("stage6a_rf_roc.png")
plt.show()


In [ ]:
# STAGE 6b: Bidirectional LSTM -- training curves (session-level model)
# trained.lstm.history_ is populated by BiLSTMClassifier.fit() during stage6_train_models above.
hist = trained.lstm.history_
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(range(1, len(hist["loss"]) + 1), hist["loss"], marker="o", color=PALETTE[1])
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("training loss")
axes[0].set_title(f"Stage 6b - LSTM training loss ({config.LSTM_EPOCHS} epochs)")

axes[1].plot(range(1, len(hist["accuracy"]) + 1), hist["accuracy"], marker="o", color=PALETTE[2])
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("training accuracy")
axes[1].set_title("Stage 6b - LSTM training accuracy")

plt.tight_layout()
savefig("stage6b_lstm_training.png")
plt.show()


In [ ]:
# STAGE 6c: XGBoost -- flow-level evaluation on the validation split
proba_xgb_val = models.predict_proba_xgb(trained.xgb, X_val[fsets["xgb"]])
pred_xgb_val = np.array(CLASSES)[proba_xgb_val.argmax(axis=1)]
xgb_macro_f1 = metrics.macro_f1(y_val_true, pred_xgb_val)
print(f"XGBoost validation macro-F1: {xgb_macro_f1:.4f}")


In [ ]:
# STAGE 6: classifier comparison table + grouped bar chart
# LSTM is evaluated at its native granularity (session-level).
val_seqs_for_cmp = events.build_session_sequences(val_df)
val_sids_for_cmp = list(val_seqs_for_cmp.keys())
val_seq_idx_for_cmp = [events.sequence_to_indices(val_seqs_for_cmp[s]) for s in val_sids_for_cmp]
val_session_labels = sessions.session_summary(val_df).set_index("session_id")["label"].loc[val_sids_for_cmp]
proba_lstm_val = trained.lstm.predict_proba(val_seq_idx_for_cmp)
pred_lstm_val = np.array(CLASSES)[proba_lstm_val.argmax(axis=1)]
lstm_macro_f1 = metrics.macro_f1(val_session_labels.values, pred_lstm_val)
lstm_acc = float((pred_lstm_val == val_session_labels.values).mean())

comparison_table = pd.DataFrame([
    {"model": "Random Forest", "features": "Group A+B (flow-level)",
     "macro_f1": rf_macro_f1, "accuracy": float((pred_rf_val == y_val_true).mean())},
    {"model": "BiLSTM", "features": "token sequences (session-level)",
     "macro_f1": lstm_macro_f1, "accuracy": lstm_acc},
    {"model": "XGBoost", "features": "Group B+D (flow-level)",
     "macro_f1": xgb_macro_f1, "accuracy": float((pred_xgb_val == y_val_true).mean())},
])
display(comparison_table)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(comparison_table))
width = 0.35
ax.bar(x - width / 2, comparison_table["macro_f1"], width, label="Macro-F1", color=PALETTE[0])
ax.bar(x + width / 2, comparison_table["accuracy"], width, label="Accuracy", color=PALETTE[1])
ax.set_xticks(x); ax.set_xticklabels(comparison_table["model"])
ax.set_ylim(0, 1)
ax.set_title("Stage 6 - Classifier comparison (validation)")
ax.legend()
plt.tight_layout()
savefig("stage6_comparison.png")
plt.show()


## Stage 8 · FP-Growth + PrefixSpan Pattern Mining

**Novel contribution**: FP-Growth mines *unordered* co-occurrence itemsets
("which tokens tend to appear together in a session"); PrefixSpan mines
*ordered* sequential patterns under a temporal `max_gap` constraint ("in
what order, with what timing"). These are complementary, not redundant, and
both feed the sequential-pattern prior `SP_t` used in Stage 7 fusion. Mined
from **training sessions only**, on **original (non-SMOTE) tokens only**.

`min_support=0.02` here (not `0.15`/`0.10`): earlier runs against the real
dataset showed attack-specific token combinations (`DOS_INDICATOR`,
`SCAN_ACTIVITY`) sit under ~3% of training flows, so a 0.15/0.10 bar let
through only near-universal BENIGN-browsing itemsets and mined nothing
attack-specific -- see `config.FPGROWTH_MIN_SUPPORT`'s comment.


In [ ]:
# STAGE 8: FP-Growth + PrefixSpan pattern mining (training sessions only)
train_seqs_raw = events.build_session_sequences(train_df)
train_tokens_only = {sid: [t for t, _ in seq] for sid, seq in train_seqs_raw.items()}

fp_patterns = pattern_mining.mine_fp_growth(train_tokens_only)
seq_patterns = pattern_mining.mine_prefixspan(train_seqs_raw)

print(f"FP-Growth: {len(fp_patterns)} frequent itemsets | PrefixSpan: {len(seq_patterns)} sequential patterns")

# OUTPUT: FP-Growth support bar chart (top 15) + PrefixSpan pattern horizontal bar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_fp = fp_patterns.sort_values("support", ascending=False).head(15)
fp_labels = [", ".join(sorted(s))[:40] for s in top_fp["itemsets"]]
axes[0].barh(fp_labels[::-1], top_fp["support"].values[::-1], color=PALETTE[5])
axes[0].set_xlabel("support"); axes[0].set_title(f"Stage 8 - Top FP-Growth itemsets (min_support={config.FPGROWTH_MIN_SUPPORT})")

top_seq = sorted(seq_patterns, key=lambda ps: ps[1], reverse=True)[:15]
seq_labels = [" -> ".join(p)[:40] for p, _ in top_seq]
seq_supports = [s for _, s in top_seq]
axes[1].barh(seq_labels[::-1], seq_supports[::-1], color=PALETTE[6])
axes[1].set_xlabel("temporal support"); axes[1].set_title(f"Stage 8 - Top PrefixSpan patterns (min_support={config.PREFIXSPAN_MIN_SUPPORT})")

plt.tight_layout()
savefig("stage8_pattern_mining.png")
plt.show()


## Stage 9-10 · Temporal Attack-State Graph & Sequence Consistency

Nodes are the 10 behavioral tokens themselves (data-driven, **not**
predefined MITRE ATT&CK stages, since CIC-IDS2017 does not contain every
kill-chain stage). Edge weights are an online EMA estimate of transition
probability, `W_t = rho*W_{t-1} + (1-rho)*W_new`, `rho=0.9`, built from
**training sessions only**, in chronological order. `lambda` (TC_t's
temporal decay constant) is selected on the **validation split** by
maximising ROC-AUC of TC_t as an attack/benign separator -- never assumed.


In [ ]:
# STAGE 9: build the attack-state graph from TRAINING sessions only
graph = attack_graph.AttackStateGraph().build_from_training(train_seqs_raw)
mean_train_gap = risk.compute_mean_interevent_time(train_seqs_raw)

lam0 = config.TC_LAMBDA_CANDIDATES[len(config.TC_LAMBDA_CANDIDATES) // 2]
val_sessions, val_seqs = pipeline.build_session_level_dataset(
    val_df, X_val, trained, fsets, fp_patterns, seq_patterns, graph, lam0, mean_train_gap
)
val_is_attack = val_sessions["is_attack"].to_dict()

# STAGE 10: select lambda on the validation split, then recompute TC_t with it
lam, tc_auc = attack_graph.select_lambda(val_seqs, graph, val_is_attack)
val_sessions["TC_t"] = [attack_graph.compute_tc_t(val_seqs[s], graph, lam) for s in val_sessions.index]
print(f"Selected lambda={lam}, validation TC_t ROC-AUC={tc_auc:.4f}")

train_sessions, _ = pipeline.build_session_level_dataset(
    train_df, X_train, trained, fsets, fp_patterns, seq_patterns, graph, lam, mean_train_gap
)
test_sessions, test_seqs = pipeline.build_session_level_dataset(
    test_df, X_test, trained, fsets, fp_patterns, seq_patterns, graph, lam, mean_train_gap
)
print("session-level datasets:", len(train_sessions), len(val_sessions), len(test_sessions))


In [ ]:
# OUTPUT: networkx graph visualisation (edge weight -> colour/width) + TC_t boxplot by class
import networkx as nx

G = graph.graph  # lazily-built networkx view of the current EMA weight matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

pos = nx.spring_layout(G, seed=SEED, weight="weight")
edge_weights = np.array([G[u][v]["weight"] for u, v in G.edges()])
edge_widths = 1.0 + 4.0 * (edge_weights / (edge_weights.max() + 1e-9))
nx.draw_networkx_nodes(G, pos, ax=axes[0], node_color=PALETTE[7], node_size=1400)
nx.draw_networkx_labels(G, pos, ax=axes[0], font_size=7)
edges = nx.draw_networkx_edges(
    G, pos, ax=axes[0], width=edge_widths, edge_color=edge_weights,
    edge_cmap=plt.cm.viridis, arrows=True, arrowsize=12, connectionstyle="arc3,rad=0.08",
)
axes[0].set_title("Stage 9 - Attack-state graph (EMA edge weights, training)")
axes[0].axis("off")

tc_by_class = val_sessions[["TC_t", "label"]]
order = [c for c in CLASSES if (tc_by_class["label"] == c).any()]
sns.boxplot(data=tc_by_class, x="label", y="TC_t", order=order, ax=axes[1],
            palette=[CLASS_COLOR[c] for c in order])
axes[1].set_title("Stage 10 - TC_t distribution by class (validation)")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
savefig("stage9_10_graph.png")
plt.show()


## Stage 7 · Adaptive Evidence Fusion

`R_t = w_A*P_A + w_B*P_B + w_C*P_C + w_S*SP_t + w_T*TC_t`

Since RF/XGBoost are flow-level and the LSTM is session-level, flow
probabilities are mean-pooled per session (`pipeline._flow_probs_to_session_probs`,
used inside `build_session_level_dataset` above) before fusion. Weights are
grid-searched **on the validation split only** to maximise macro-F1, then
**frozen** for every test session -- a rolling/live-F1 scheme would need
test-time labels, which is a leakage bug.


In [ ]:
# STAGE 7: calibrate fusion weights on validation, freeze, apply everywhere
fusion_weights, val_macro_f1_fused = pipeline.stage7_calibrate_fusion(val_sessions)
print("Frozen fusion weights:", fusion_weights.as_dict())
print(f"Validation macro-F1 at these weights: {val_macro_f1_fused:.4f}")

train_sessions = pipeline.add_fused_predictions(train_sessions, fusion_weights)
val_sessions = pipeline.add_fused_predictions(val_sessions, fusion_weights)
test_sessions = pipeline.add_fused_predictions(test_sessions, fusion_weights)

# OUTPUT: fusion weight pie chart + fused probability histogram (benign vs attack)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
w = fusion_weights.as_dict()
axes[0].pie(w.values(), labels=[f"{k} ({v:.2f})" for k, v in w.items()],
            colors=sns.color_palette("tab20", len(w)), startangle=90)
axes[0].set_title("Stage 7 - Frozen fusion weights")

benign_mask = val_sessions["label"] == "BENIGN"
axes[1].hist(val_sessions.loc[benign_mask, "R_max"], bins=25, alpha=0.6, label="BENIGN", color=PALETTE[0])
axes[1].hist(val_sessions.loc[~benign_mask, "R_max"], bins=25, alpha=0.6, label="attack", color=PALETTE[3])
axes[1].set_xlabel("fused R_max (top-class probability)")
axes[1].set_title("Stage 7 - Fused probability distribution (validation)")
axes[1].legend()

plt.tight_layout()
savefig("stage7_fusion.png")
plt.show()

print("\nSample fused probability table (top-5 classes, one example session):")
example_sid = val_sessions.index[0]
top5 = val_sessions.loc[example_sid, [f"R_{c}" for c in CLASSES]].sort_values(ascending=False).head(5)
display(top5.rename("R_t"))


## Stage 11 · Adaptive Risk Meta-Learner

**Novel contribution**: a logistic-regression meta-learner combines
`R_t` (15-dim), `SP_t`, `TC_t`, `G_w` (graph centrality/consistency proxy)
and `1/dt_norm` into a single `Risk_t in [0,1]`, fit on the **validation
split** -- coefficients are learned, not hand-picked. Three-tier alerting:
BENIGN (`Risk_t < 0.35`), SUSPICIOUS (`0.35 <= Risk_t < 0.75`), ATTACK
(`Risk_t >= 0.75`) (`config.RISK_THRESHOLD_SUSPICIOUS`/`_ATTACK`).


In [ ]:
# STAGE 11: fit the risk meta-learner on validation, apply everywhere
risk_model = pipeline.stage11_train_risk_model(val_sessions)

train_sessions = pipeline.apply_risk_model(train_sessions, risk_model)
val_sessions = pipeline.apply_risk_model(val_sessions, risk_model)
test_sessions = pipeline.apply_risk_model(test_sessions, risk_model)

print("Test alert-tier distribution:")
print(test_sessions["tier"].value_counts())

# OUTPUT: risk histogram (benign vs attack, threshold lines) + alert tier pie + meta-learner ROC
from sklearn.metrics import roc_curve as _roc_curve, roc_auc_score as _roc_auc_score

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
benign_mask = val_sessions["label"] == "BENIGN"
axes[0].hist(val_sessions.loc[benign_mask, "risk"], bins=25, alpha=0.6, label="BENIGN", color=PALETTE[0])
axes[0].hist(val_sessions.loc[~benign_mask, "risk"], bins=25, alpha=0.6, label="attack", color=PALETTE[3])
axes[0].axvline(config.RISK_THRESHOLD_SUSPICIOUS, color="orange", linestyle="--", label="SUSPICIOUS threshold")
axes[0].axvline(config.RISK_THRESHOLD_ATTACK, color="red", linestyle="--", label="ATTACK threshold")
axes[0].set_xlabel("Risk_t"); axes[0].set_title("Stage 11 - Risk distribution (validation)")
axes[0].legend(fontsize=8)

tier_counts = test_sessions["tier"].value_counts()
axes[1].pie(tier_counts.values, labels=tier_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("tab20", len(tier_counts)))
axes[1].set_title("Stage 11 - Test alert-tier distribution")

fpr, tpr, _ = _roc_curve(val_sessions["is_attack"], val_sessions["risk"])
meta_auc = _roc_auc_score(val_sessions["is_attack"], val_sessions["risk"])
axes[2].plot(fpr, tpr, color=PALETTE[2], label=f"AUC={meta_auc:.3f}")
axes[2].plot([0, 1], [0, 1], "k--", linewidth=0.8)
axes[2].set_xlabel("False Positive Rate"); axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("Stage 11 - Risk meta-learner ROC (validation)")
axes[2].legend()

plt.tight_layout()
savefig("stage11_risk.png")
plt.show()


## Evaluation Metrics (Test Set)

Primary metric is **Macro-F1** (unweighted mean across all 15 classes) plus
per-class F1 and false-positive rate. Accuracy is reported last, and with
caution -- BENIGN dominance makes it an uninformative headline number.
Heartbleed and Infiltration are always reported per-class, never merged
away.


In [ ]:
report = metrics.per_class_report(test_sessions["label"], test_sessions["predicted_class"])
display(report)

test_macro_f1 = metrics.macro_f1(test_sessions["label"], test_sessions["predicted_class"])
test_fpr = metrics.false_positive_rate(test_sessions["label"], test_sessions["predicted_class"])
print(f"Test Macro-F1: {test_macro_f1:.4f}   Test FPR: {test_fpr:.4f}")

pt, lo, hi = metrics.bootstrap_ci(
    test_sessions["label"].values, test_sessions["predicted_class"].values, metrics.macro_f1, n_boot=1000
)
print(f"Macro-F1 95% bootstrap CI: {pt:.4f} [{lo:.4f}, {hi:.4f}]")


## Stage 12 · Streaming Evaluation

Simulated streaming over the test set (Days 4-7), 60s tumbling windows /
10s stride. **Model weights are frozen** -- this is streaming *evaluation*
of pre-trained static models, never online learning (the only thing that
may still adapt mid-stream is the attack-state graph's EMA edge weights via
`attack_graph.update_edge_streaming`, which this evaluation does not invoke,
to keep the headline metrics reproducible).


In [ ]:
# STAGE 12: streaming evaluation
summary_for_stream = test_sessions.reset_index().rename(columns={"index": "session_id"})[
    ["session_id", "start_time", "end_time", "n_flows", "label"]
]
risk_lookup = test_sessions["risk"].to_dict()
pred_lookup = test_sessions["predicted_class"].to_dict()

# Wrap the score function to also record per-session wall-clock latency, so
# the per-window latency/throughput plots below reflect real measured time
# (streaming.simulate_streaming only reports the whole-run aggregate).
import time as _time
_latency_by_sid = {}

def score_session(sid):
    t0 = _time.perf_counter()
    r = risk_lookup[sid]
    _latency_by_sid[sid] = _time.perf_counter() - t0
    return r

stream_result = streaming.simulate_streaming(summary_for_stream, score_session)
print(f"Throughput: {stream_result.throughput_events_per_sec:.1f} events/sec")
print(f"Mean latency: {stream_result.mean_latency_ms_per_event:.4f} ms/event")
print(streaming.early_detection_stats(stream_result))

# Per-window metrics: join per_session rows back to test_sessions for predicted_class,
# then aggregate by window_start (dropping the "missed"/NaT rows scored outside any window).
per_session = stream_result.per_session.copy()
per_session["predicted_class"] = per_session["session_id"].map(pred_lookup)
per_session["n_flows"] = per_session["session_id"].map(summary_for_stream.set_index("session_id")["n_flows"])
per_session["latency_ms"] = per_session["session_id"].map(_latency_by_sid) * 1000.0
windowed = per_session.dropna(subset=["window_start"])

window_stats = []
for w_start, g in windowed.groupby("window_start"):
    f1 = metrics.macro_f1(g["label"], g["predicted_class"]) if g["label"].nunique() > 1 else np.nan
    window_stats.append({
        "window_start": w_start,
        "macro_f1": f1,
        "latency_ms": g["latency_ms"].mean(),
        "throughput": g["n_flows"].sum() / max(g["latency_ms"].sum() / 1000.0, 1e-9),
        "alert_rate": g["alert"].mean(),
    })
window_df = pd.DataFrame(window_stats).sort_values("window_start")

# OUTPUT (2x2 subplot): macro-F1, latency, throughput, alert rate vs window_start
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(window_df["window_start"], window_df["macro_f1"], marker="o", color=PALETTE[0])
axes[0, 0].axhline(window_df["macro_f1"].mean(), color="gray", linestyle="--", label="mean F1")
axes[0, 0].set_title("Macro-F1 vs window_start"); axes[0, 0].legend(); axes[0, 0].tick_params(axis="x", rotation=30)

axes[0, 1].plot(window_df["window_start"], window_df["latency_ms"], marker="o", color=PALETTE[1])
axes[0, 1].set_title("Inference latency (ms/session) vs window_start"); axes[0, 1].tick_params(axis="x", rotation=30)

axes[1, 0].plot(window_df["window_start"], window_df["throughput"], marker="o", color=PALETTE[2])
axes[1, 0].set_title("Throughput (sessions/sec) vs window_start"); axes[1, 0].tick_params(axis="x", rotation=30)

axes[1, 1].plot(window_df["window_start"], window_df["alert_rate"], marker="o", color=PALETTE[3])
axes[1, 1].set_title("Alert rate vs window_start"); axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig("stage12_streaming.png")
plt.show()


## Stage 13 · Explainability & Evidence Chain

For each ATTACK/SUSPICIOUS alert, three complementary explanation
modalities feed one analyst-facing report:
**(a)** TreeSHAP for RF/XGBoost (exact feature attributions),
**(b)** Gradient x Input saliency over the LSTM's token positions,
**(c)** the graph path with its learned edge weights.


In [ ]:
# STAGE 13a/13b: TreeSHAP for RF and XGBoost, on a 500-flow test sample
import shap

SHAP_SAMPLE_N = min(500, len(X_test))
shap_sample_idx = X_test.sample(n=SHAP_SAMPLE_N, random_state=SEED).index

rf_explainer = shap.TreeExplainer(trained.rf)
rf_shap_values = rf_explainer.shap_values(X_test.loc[shap_sample_idx, fsets["rf"]])

xgb_explainer = shap.TreeExplainer(trained.xgb)
xgb_shap_values = xgb_explainer.shap_values(X_test.loc[shap_sample_idx, fsets["xgb"]])

def _mean_abs_shap(shap_values, feature_names, top_k=15):
    '''Handles both list-of-arrays (per class) and (n, features, classes) shap outputs.'''
    if isinstance(shap_values, list):
        arr = np.mean([np.abs(c) for c in shap_values], axis=0)
    elif np.ndim(shap_values) == 3:
        arr = np.abs(shap_values).mean(axis=2)
    else:
        arr = np.abs(shap_values)
    mean_abs = arr.mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:top_k]
    return pd.Series(mean_abs[order], index=np.array(feature_names)[order])

rf_importance = _mean_abs_shap(rf_shap_values, fsets["rf"])
xgb_importance = _mean_abs_shap(xgb_shap_values, fsets["xgb"])

plt.figure(figsize=(9, 6))
shap.summary_plot(rf_shap_values, X_test.loc[shap_sample_idx, fsets["rf"]], show=False, plot_type="bar")
plt.title("Stage 13a - RF TreeSHAP summary")
plt.tight_layout()
savefig("stage13a_shap_rf.png")
plt.show()

plt.figure(figsize=(9, 6))
shap.summary_plot(xgb_shap_values, X_test.loc[shap_sample_idx, fsets["xgb"]], show=False, plot_type="bar")
plt.title("Stage 13b - XGBoost TreeSHAP summary")
plt.tight_layout()
savefig("stage13b_shap_xgb.png")
plt.show()

# OUTPUT: side-by-side feature importance bars for RF and XGBoost
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(rf_importance.index[::-1], rf_importance.values[::-1], color=PALETTE[0])
axes[0].set_title("RF - mean |SHAP|"); axes[0].set_xlabel("mean |SHAP value|")
axes[1].barh(xgb_importance.index[::-1], xgb_importance.values[::-1], color=PALETTE[1])
axes[1].set_title("XGBoost - mean |SHAP|"); axes[1].set_xlabel("mean |SHAP value|")
plt.tight_layout()
savefig("stage13_shap_importance.png")
plt.show()


In [ ]:
# STAGE 13c: Graph Path Evidence for one ATTACK-tier test session
attack_alerts = test_sessions[test_sessions["tier"] == "ATTACK"]

if len(attack_alerts) == 0:
    print("No ATTACK-tier sessions in the test set for this run/dataset -- "
          "13c/13d skipped (this can legitimately happen on a small synthetic "
          "smoke-test dataset; on the real dataset check Stage 1's zero-shot "
          "warning if it also happens there).")
    evidence_sid = None
else:
    evidence_sid = attack_alerts.index[0]
    row = test_sessions.loc[evidence_sid]
    seq = test_seqs[evidence_sid]
    seq_idx = events.sequence_to_indices(seq)
    path = attack_graph.graph_path_evidence(seq, graph)

    # OUTPUT (side-by-side): token sequence timeline step-plot + graph subgraph with highlighted path
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    tok_ids = [TOKEN2ID[t] for t, _ in seq]
    axes[0].step(range(len(tok_ids)), tok_ids, where="mid", marker="o", color=PALETTE[3])
    axes[0].set_yticks(range(len(VOCAB))); axes[0].set_yticklabels(VOCAB, fontsize=8)
    axes[0].set_xlabel("flow index within session")
    axes[0].set_title(f"Token sequence - session {evidence_sid} ({row['label']})")

    path_nodes = sorted({n for n, _, _ in path} | {n for _, n, _ in path})
    subG = graph.graph.subgraph(path_nodes)
    pos = nx.spring_layout(subG, seed=SEED, weight="weight")
    nx.draw_networkx_nodes(subG, pos, ax=axes[1], node_color=PALETTE[7], node_size=1600)
    nx.draw_networkx_labels(subG, pos, ax=axes[1], font_size=8)
    nx.draw_networkx_edges(subG, pos, ax=axes[1], edge_color="lightgray", arrows=True)
    path_edges = [(s, d) for s, d, _ in path]
    nx.draw_networkx_edges(subG, pos, ax=axes[1], edgelist=path_edges, edge_color="red", width=2.5, arrows=True)
    edge_labels = {(s, d): f"{w:.2f}" for s, d, w in path}
    nx.draw_networkx_edge_labels(subG, pos, ax=axes[1], edge_labels=edge_labels, font_size=7)
    axes[1].set_title("Graph path evidence (red = path taken)")
    axes[1].axis("off")

    plt.tight_layout()
    savefig("stage13c_graph_evidence.png")
    plt.show()


In [ ]:
# STAGE 13d: full analyst evidence-chain report
if evidence_sid is not None:
    X_row_rf = X_test.loc[test_df[test_df["session_id"] == evidence_sid].index[:1], fsets["rf"]]
    X_row_xgb = X_test.loc[test_df[test_df["session_id"] == evidence_sid].index[:1], fsets["xgb"]]
    predicted_idx = config.CLASS_TO_IDX[row["predicted_class"]]

    rf_top = explainability.explain_tree_model(trained.rf, X_row_rf, target_class_idx=None)
    xgb_top = explainability.explain_tree_model(trained.xgb, X_row_xgb, target_class_idx=None)
    lstm_attrib = explainability.explain_lstm_sequence(trained.lstm, seq_idx, predicted_idx)

    print("=" * 60)
    print(f"EVIDENCE CHAIN REPORT -- Session {evidence_sid}")
    print("=" * 60)
    print(f"True class      : {row['label']}")
    print(f"Predicted class : {row['predicted_class']}")
    print(f"Risk_t          : {row['risk']:.3f}")
    print(f"Alert tier      : {row['tier']}")
    print(f"SP_t (pattern)  : {row['SP_t']:.3f}")
    print(f"TC_t (consist.) : {row['TC_t']:.3f}")
    print(f"G_w (centrality): {row['G_w']:.3f}")
    print()
    print(f"[A] Top-{config.SHAP_TOP_K} TreeSHAP Features (RF):")
    for feat, val in rf_top:
        marker = "^" if val > 0 else "v"
        print(f"    {marker} {feat:<30s} |SHAP|={abs(val):.4f}")
    print()
    print("[B] Behavioral Token Sequence:")
    print("    " + " -> ".join(t for t, _ in seq))
    print()
    print("[C] Graph Path Evidence (high-weight transitions):")
    for s, d, w in sorted(path, key=lambda e: e[2], reverse=True)[:5]:
        print(f"    {s} -> {d}  w={w:.2f}")
    print()
    print("[D] Recommended Action:")
    print(f"    Block source IP, audit destination for {row['predicted_class']}.")
    print("=" * 60)


## Final Summary

Results table across all models and stages, plus a comparison bar chart.


In [ ]:
# FINAL SUMMARY
final_results = pd.DataFrame([
    {"Model": "Random Forest", "Macro F1": rf_macro_f1},
    {"Model": "BiLSTM", "Macro F1": lstm_macro_f1},
    {"Model": "XGBoost", "Macro F1": xgb_macro_f1},
    {"Model": "Fused (Stage 7)", "Macro F1": val_macro_f1_fused},
    {"Model": "Risk Meta-Learner (AUC)", "Macro F1": meta_auc},
    {"Model": "Test set (fused+risk, final)", "Macro F1": test_macro_f1},
    {"Model": "Streaming Mean F1", "Macro F1": window_df["macro_f1"].mean()},
])
display(final_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(final_results["Model"], final_results["Macro F1"], color=sns.color_palette("tab20", len(final_results)))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Final results summary")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig("final_results_summary.png")
plt.show()

print("All 13 stages complete. Ready for IEEE submission.")
